# Fine-tune Moirai-MoE-base on S&P 500 v3 -- all variables feed the model, only 3 are forecasted

This is a third run of the fine-tuning pipeline, built in response to v2's result: fine-tuning made
*every* metric worse than zero-shot (Close MAPE 2.8% -> 3.2%, Return/LogReturn MAE both ~+38%), and
the loss curve showed a classic overfitting signature (train loss fell smoothly the whole time while
validation loss went noisy/plateaued early, with a widening train/val gap). v3 changes four things:

1. **All engineered variables now feed the model** -- Open, High, Low, Volume, Momentum, MovingAverage,
   and cyclical (sin/cos) Day-of-week/Month-of-year encodings are fed in as context-only covariates
   (`past_feat_dynamic_real`). Only **Close, Return, LogReturn are ever forecasted** -- the covariates
   inform the prediction without being predicted themselves. This required adding real covariate
   support to uni2ts's CSV data builder (`wide_multivariate_covariates` dataset type); the model-side
   transform pipeline and `MoiraiMoEForecast` already supported `past_feat_dynamic_real` natively.
2. **`finetune_pattern=freeze_ffn` instead of `full`** -- v2's full fine-tune updated all 935M params on
   one narrow, near-random-walk series, which is exactly the overfitting setup the loss curve showed.
   `freeze_ffn` freezes the (in Moirai-MoE, MoE-routed) FFN blocks -- the bulk of the parameter count and
   the most overfitting-prone part -- while still letting attention/projection layers adapt to the new
   covariate-enriched input.
3. **Real early stopping restored** (`patience=3`, matching the default -- v2 had this neutralized).
   The evidence from both v1 and v2 is that validation loss plateaus/gets noisy early; letting the
   trainer find that point automatically is more principled than betting on a fixed epoch count.
4. **`data.distance=16` instead of `5`** -- v2's training windows were 507/512 days identical to their
   neighbors (99% overlap); this reduces redundancy so each epoch sees more genuinely distinct windows.

**Requires a GPU runtime**: Runtime -> Change runtime type -> T4 GPU (or better).

Note: with 10 covariates added, each packed training sample is roughly 4x longer than v2's (target-only)
samples, so `train_dataloader.batch_size` is reduced to `4` (from v2's `16`) as an untested-on-real-GPU
precaution against OOM -- if you have headroom to spare, you can try raising it; if you hit OOM, lower it
further (e.g. `2`).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> select a GPU, then re-run this cell.")

## 1. Clone the repo and install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/Agrim-Nuware/MOIRAI-CODE.git"
if not os.path.isdir("repo"):
    !git clone $REPO_URL repo
%cd repo

In [ ]:
!pip install -q -e '.[notebook]'
!pip install -q bitsandbytes yfinance

# uni2ts pins numpy~=1.26, which downgrades Colab's preinstalled numpy 2.x.
# Colab's preinstalled pandas is built against numpy 2.x, so once numpy is
# downgraded the two are ABI-incompatible ("numpy.dtype size changed").
# Force-reinstall a matching pandas, then restart the runtime so every
# already-imported module (numpy got pulled in transitively by torch above)
# reloads consistently. This cell intentionally crashes/restarts the kernel --
# that's expected, not an error. After it restarts, just continue running
# from the next cell (installed packages and cloned files are unaffected).
!pip install -q --force-reinstall "numpy<2" "pandas>=2.0,<2.3"
import os

os.kill(os.getpid(), 9)

**The cell above deliberately restarts the Colab runtime** (to fix a numpy/pandas
version mismatch). You'll see a "session crashed" / "automatically restarted" notice --
that's expected. Once it restarts, just continue running the cells below in order;
you do **not** need to re-run the clone or pip install cells.

In [ ]:
%cd /content/repo
import numpy as np
import pandas as pd

print("numpy:", np.__version__, "| pandas:", pd.__version__)

In [ ]:
with open(".env", "w") as f:
    f.write("CUSTOM_DATA_PATH=dataset/uni2ts_storage\n")
print(open(".env").read())

## 2. Download 20 years of S&P 500 data, engineer features, split at 2024-10-01

In [ ]:
!python dataset/sp500/prepare_v3_data.py

In [ ]:
import json
import pandas as pd

with open("dataset/sp500/split_info_v3.json") as f:
    split_info = json.load(f)

trainval_df = pd.read_csv("dataset/sp500/sp500_v3_model_input_trainval.csv", index_col=0, parse_dates=True)
train_length = split_info["train_length"]
date_offset = trainval_df.index[train_length - 1].strftime("%Y-%m-%d")
target_columns = ",".join(split_info["target_columns"])
covariate_columns = ",".join(split_info["covariate_columns"])

print("target columns (forecasted):", target_columns)
print("covariate columns (context only):", covariate_columns)
print("train_length:", train_length)
print("lightning val offset:", split_info["lightning_val_offset"])
print("lightning val length:", split_info["lightning_val_length"])
print("final test length (>= cutoff):", split_info["final_test_len"])
print("date_offset for CSV builder:", date_offset)

## 3. Build the uni2ts HF-format dataset (target + covariate split)

In [ ]:
!python -m uni2ts.data.builder.simple SP500V3 dataset/sp500/sp500_v3_model_input_trainval.csv \
  --dataset_type wide_multivariate_covariates \
  --target_columns "{target_columns}" \
  --covariate_columns "{covariate_columns}" \
  --date_offset "{date_offset}" --freq B

## 4. Fine-tune Moirai-MoE-base (freeze_ffn, GPU, fp16 + 8-bit AdamW, real early stopping)

`data.mode=MC` / `val_data.mode=MC` route through the new covariate-aware dataset builder. Unlike v2,
there is no `trainer.callbacks.2.patience` override here -- the default `patience=3` from
`cli/conf/finetune/default.yaml` applies, so training stops automatically once validation loss stops
improving. `trainer.max_epochs=40` is just a generous safety ceiling, not the expected stopping point.

In [ ]:
lightning_val_offset = split_info["lightning_val_offset"]
lightning_val_length = split_info["lightning_val_length"]

!python -m cli.train \
  -cp conf/finetune \
  exp_name=sp500_v3_full_finetune \
  run_name=run1 \
  tf32=false \
  model=moirai_moe_1.0_R_base \
  model.patch_size=16 \
  model.context_length=512 \
  model.prediction_length=32 \
  model.num_training_steps=2500 \
  model.num_warmup_steps=50 \
  model.finetune_pattern=freeze_ffn \
  model.use_8bit_adam=true \
  model.lr=1e-5 \
  data=sp500 \
  data.dataset=SP500V3 \
  data.patch_size=16 \
  data.context_length=512 \
  data.prediction_length=32 \
  data.mode=MC \
  data.train_length={train_length} \
  data.distance=16 \
  val_data=sp500 \
  val_data.dataset=SP500V3_eval \
  val_data.patch_size=16 \
  val_data.context_length=512 \
  val_data.prediction_length=32 \
  val_data.mode=MC \
  val_data.offset={lightning_val_offset} \
  val_data.eval_length={lightning_val_length} \
  val_data.distance=32 \
  trainer.max_epochs=40 \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  trainer.precision=16-mixed \
  +trainer.log_every_n_steps=10 \
  train_dataloader.batch_size=4 \
  train_dataloader.num_workers=0 \
  val_dataloader.batch_size=2 \
  val_dataloader.num_workers=0

## 5. Evaluate: zero-shot vs fine-tuned, on the held-out region (>= 2024-10-01)

Both models are evaluated with the same covariates available as context -- the comparison is purely
about whether fine-tuning (with covariates + freeze_ffn + early stopping) improved on zero-shot, not
about whether covariates alone help (zero-shot also gets them here).

In [ ]:
!python dataset/sp500/evaluate_finetuned_v3.py \
  --context_length 512 --prediction_length 32 --num_samples 100

In [ ]:
from IPython.display import Image, display

print("Forecast comparison (Close price -- direct vs reconstructed from LogReturn):")
display(Image("dataset/sp500/results_v3_forecast_plot.png"))
print("\nError comparison by variate:")
display(Image("dataset/sp500/results_v3_metrics_bar.png"))
print("\nFine-tuning loss curve:")
display(Image("dataset/sp500/results_v3_loss_curve.png"))

In [ ]:
import json

with open("dataset/sp500/results_v3_metrics.json") as f:
    results = json.load(f)

zs, ft = results["zero_shot"], results["fine_tuned"]
print(f"{'metric':<22} {'zero-shot':>14} {'fine-tuned':>14}")
print(f"{'Close MAPE':<22} {zs['Close']['mape']:>13.2f}% {ft['Close']['mape']:>13.2f}%")
print(f"{'Close(recon) MAPE':<22} {zs['Close_reconstructed']['mape']:>13.2f}% {ft['Close_reconstructed']['mape']:>13.2f}%")
print(f"{'Return MAE':<22} {zs['Return']['mae']:>14.5f} {ft['Return']['mae']:>14.5f}")
print(f"{'LogReturn MAE':<22} {zs['LogReturn']['mae']:>14.5f} {ft['LogReturn']['mae']:>14.5f}")

## 6. (Optional) Save results back to your GitHub repo

Uncomment and fill in a [personal access token](https://github.com/settings/tokens) if you
want to push the plots/metrics back to your repo. Skip this if you'd rather just download
the files from the Colab file browser (left sidebar).

In [ ]:
# GITHUB_TOKEN = ""  # paste a token with repo write access, or leave blank to skip
# if GITHUB_TOKEN:
#     !git add dataset/sp500/results_v3_*.png dataset/sp500/results_v3_metrics.json
#     !git commit -m "Add Colab v3 fine-tuning results"
#     !git push https://$GITHUB_TOKEN@github.com/Agrim-Nuware/MOIRAI-CODE.git HEAD:main